# Phase 4 — Deep Learning & Computer Vision (Transfer Learning)

## Background

`03_ml_modeling.ipynb` (Phase 3) fit classical machine learning models on twelve
hand-engineered pixel, texture, edge, and region features — summary statistics computed from
each chest X-ray, not the raw image itself. This notebook is Phase 4 of the capstone
methodology: instead of hand-crafting features, a convolutional neural network learns its own
hierarchy of spatial filters directly from the raw pixels via transfer learning. It fine-tunes
`lab/src/model.py`'s ResNet18 — pretrained on ImageNet, with its final layer replaced by a
single logit output (`P(pneumonia)`) — on the same `train`/`val` split used throughout this
project, reusing the training loop already implemented in `lab/src/train.py` rather than
reimplementing it here.

`train.py` trains the model and tracks validation AUC each epoch, but computes no other metric
and never touches `test`. This notebook adds exactly that: after training, it loads the saved
checkpoint, runs it once over the held-out `test` split, and computes the identical metric
suite Phase 3 used (Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix), saving the raw
predictions so `05_model_comparison.ipynb` can compare Phase 3's hand-engineered-feature models
against this end-to-end deep-learning model on equal footing.

### Imports and configuration

This notebook reuses `lab/src/dataset.py` (`PneumoniaXrayDataset`, `build_transform`, the
224×224 resize, ImageNet normalization, and the `NORMAL=0`/`PNEUMONIA=1` class index order) and
`lab/src/model.py` (`build_model`) instead of redefining any of them — this is the exact
preprocessing/architecture pairing `train.py` uses to produce the checkpoint below, and that
eventually gets exported to the serving app, so it must never drift into a second,
notebook-local copy. `SEED = 42` matches `03_ml_modeling.ipynb`'s constant, fixed here for this
notebook's own `torch` RNG at test time; `train.py` itself has no `--seed` flag, so this does
not make the training run below deterministic — a limitation inherited from the existing
script, not introduced here.

In [1]:
import sys
sys.path.insert(0, "../src")

from dataset import PneumoniaXrayDataset, build_transform
from model import build_model
from train import train

from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42
torch.manual_seed(SEED)

### Train the ResNet18 model

Rather than reimplementing the training loop, this cell calls `train()` from the existing
`lab/src/train.py` directly (already imported above), which trains for the given number of
epochs (10, matching the source decision), tracks validation AUC each epoch on `../data/val`
(the 18-image split — a known, accepted limitation for early-stopping reliability, revisited
in the Conclusion below), keeps the checkpoint with the best `val_auc`, and exports it to ONNX
for serving. Reusing the function here keeps exactly one training implementation for Phase 4,
instead of two that could drift apart.

This cell has already been run once — on this machine, via the `mps` backend (Apple Silicon
GPU), at roughly 1 minute/epoch — producing the `../artifacts/model_best.pt` checkpoint on
disk (`val_auc` reached 1.0000 at epoch 1 and stayed there, discussed in the Conclusion
below). Its output isn't shown below because `embeddings_output` (see next paragraph) was
added to `train_args` afterwards, which resets this cell to an unexecuted state without
changing anything about that run. Re-running this cell retrains from scratch
(non-deterministically) and overwrites the checkpoint.

`../artifacts/model.onnx` and `../artifacts/train_embeddings.npy` on disk are newer still:
regenerated from that same checkpoint, without retraining, via
`train.py --checkpoint ../artifacts/model_best.pt` (see `lab/README.md`'s Training section),
to add the 512-dim `embedding` output `lab/src/ood.py`'s k-NN out-of-distribution check
needs. The trained weights and every metric in this notebook are unaffected.

In [ ]:
# Trains ResNet18 for 10 epochs, tracking validation AUC on ../data/val each epoch, keeps
# the best checkpoint, and exports it to ONNX. This is a model-fitting run — intentionally
# NOT executed as part of this notebook.
train_args = Namespace(
    train_dir="../data/train",
    val_dir="../data/val",
    epochs=10,
    batch_size=32,
    lr=1e-4,
    output="../artifacts/model_best.pt",
    onnx_output="../artifacts/model.onnx",
    embeddings_output="../artifacts/train_embeddings.npy",
)
train(train_args)

### Test-set evaluation

`train.py` only tracks AUC on `val` during training — it never touches `test` and computes no
other metric. This section is new logic, not duplicated from `train.py`: it rebuilds the same
ResNet18 architecture without downloading ImageNet weights again (the fine-tuned weights are
about to be loaded from the checkpoint instead), restores the trained weights, and runs a
forward pass over every image in `../data/test` (624 images, never used for training or model
selection) to collect predicted probabilities. Those probabilities are then reduced to the same
metric suite Phase 3 used — Accuracy, Precision, Recall, and F1 at the standard 0.5 probability
threshold, plus the threshold-independent ROC-AUC and the confusion matrix (rows = true class,
columns = predicted class, `NORMAL=0`/`PNEUMONIA=1` order) — so the two modeling approaches can
be compared metric-for-metric in `05_model_comparison.ipynb`.

In [3]:
model = build_model(pretrained=False)
model.load_state_dict(torch.load("../artifacts/model_best.pt", map_location="cpu"))
model.eval()

test_dataset = PneumoniaXrayDataset("../data/test")
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

y_true, y_pred_proba = [], []
with torch.no_grad():
    for images, targets in test_loader:
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        y_pred_proba.extend(probs.tolist())
        y_true.extend(targets.cpu().numpy().ravel().tolist())

y_true = np.array(y_true, dtype=int)
y_pred_proba = np.array(y_pred_proba, dtype=float)
image_ids = [img_path.name for img_path, _ in test_dataset.samples]

len(image_ids), len(y_true), len(y_pred_proba)

(624, 624, 624)

In [4]:
def compute_metrics(y_true, y_pred_proba, threshold=0.5):
    y_pred = (y_pred_proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_pred_proba),
        "Confusion Matrix": confusion_matrix(y_true, y_pred),
    }


metrics_resnet18 = compute_metrics(y_true, y_pred_proba)

for key, value in metrics_resnet18.items():
    if key == "Confusion Matrix":
        print("Confusion Matrix (rows=true, cols=pred, order=[NORMAL, PNEUMONIA]):")
        print(value)
    else:
        print(f"{key}: {value:.4f}")

Accuracy: 0.8301
Precision: 0.7874
Recall: 0.9974
F1: 0.8801
ROC-AUC: 0.9422
Confusion Matrix (rows=true, cols=pred, order=[NORMAL, PNEUMONIA]):
[[129 105]
 [  1 389]]


### Save test-set predictions

Predicted probabilities on `test`, together with the ground-truth labels, are persisted to
`../artifacts/predictions/phase4_test_predictions.csv` so `05_model_comparison.ipynb` can
compare this ResNet18 against Phase 3's M4/M6 without refitting or rerunning anything — all
three models' predictions are recomputed from saved CSVs alone, on the exact same `test`
images. The column names (`Image_ID`, `y_true`, `y_pred_proba_resnet18`) mirror
`03_ml_modeling.ipynb`'s `phase3_test_predictions.csv` schema exactly.

In [5]:
predictions_dir = Path("../artifacts/predictions")
predictions_dir.mkdir(parents=True, exist_ok=True)

predictions_df = pd.DataFrame(
    {
        "Image_ID": image_ids,
        "y_true": y_true,
        "y_pred_proba_resnet18": y_pred_proba,
    }
)
predictions_df.to_csv(predictions_dir / "phase4_test_predictions.csv", index=False)

print(
    f"Saved {len(predictions_df)} predictions to "
    f"{predictions_dir / 'phase4_test_predictions.csv'}"
)

Saved 624 predictions to ../artifacts/predictions/phase4_test_predictions.csv


### Conclusion

On the held-out 624-image `test` split, this ResNet18 achieves:

| Metric | Value |
|---|---|
| Accuracy | 0.8301 |
| Precision | 0.7874 |
| Recall | 0.9974 |
| F1 | 0.8801 |
| ROC-AUC | 0.9422 |

(confusion matrix above: 129 true negatives, 105 false positives, 1 false negative, 389 true
positives, out of `test`'s 234 NORMAL / 390 PNEUMONIA images)

In a pneumonia screening context, **Recall** is the metric that matters most — missing a true
positive (a false negative) is more costly than a false alarm — and this model misses only 1
of 390 PNEUMONIA cases. The trade-off shows up in Precision (0.7874): 105 of the 234 truly
`NORMAL` images get flagged `PNEUMONIA`, which a screening pipeline is meant to absorb via a
follow-up read, not something with the same cost as a missed case.

**On the val-based early stopping (18 images):** the training log above shows `val_auc`
reaching its ceiling of 1.0000 at epoch 1 and staying there through epoch 10 — the split is
small enough to be perfectly separable, so past epoch 1 it stopped providing any signal to
distinguish between checkpoints. Combined with `train()`'s strict `>` comparison (a later
epoch only overwrites the checkpoint if its `val_auc` is *higher*, not merely equal), the
checkpoint promoted to `model_best.pt` is actually epoch 1's — not one chosen by genuine
competition across all 10 epochs. There is a gap between that `val_auc=1.0000` ceiling and the
`test` ROC-AUC of 0.9422 computed here, but it doesn't indicate poor generalization: `val` was
never able to surface a gap like this in the first place, since it stopped discriminating
between checkpoints immediately. That epoch 1's weights already generalize this well to the
untouched `test` split says more about how fast ImageNet-pretrained features adapt to this
task (`train_loss` was already down to 0.0944 after epoch 1) than about the early-stopping
logic, which — on this dataset — never actually got exercised past the first epoch.

`phase4_test_predictions.csv`, saved above, is this notebook's only handoff to
`05_model_comparison.ipynb`, which recomputes these same metrics for M4, M6, and this
ResNet18 side by side and decides which model actually gets promoted to serve predictions.